# Simple: Partial Masking with PAMOLA.CORE

**Goal**: Learn partial masking basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply position-based masking using execute()
- Compare before/after results
- Understand privacy improvements through selective masking

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/masking/
        └── 01_partial_masking_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display

print("🔍 Detecting PAMOLA installation...\n")

# Auto-detect project root (search up 6 levels for pamola_core/)
current_dir = Path.cwd()
project_root = current_dir
for level in range(6):
    if (project_root / 'pamola_core').exists():
        print(f"✓ Found pamola_core at level {level}: {project_root}")
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError("❌ Could not find pamola_core directory!")

sys.path.insert(0, str(project_root))

# Import operation-specific modules
from pamola_core.anonymization.masking.partial_masking_op import (
    PartialMaskingOperation
)
from pamola_core.utils.ops.op_data_source import DataSource
from pamola_core.utils.progress import HierarchicalProgressTracker
from pamola_core.utils.tasks.task_reporting import TaskReporter

print("✅ All imports successful!")

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records containing sensitive fields suitable for masking

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with sensitive information
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'email': [
            'john.doe@email.com', 'jane.smith@company.org', 'alice.wonder@test.net',
            'bob.builder@construct.io', 'charlie.brown@peanuts.com', 'david.jones@work.biz',
            'emma.watson@magic.uk', 'frank.castle@vigilante.com', 'grace.hopper@navy.mil',
            'henry.ford@auto.com', 'iris.chang@history.edu', 'jack.sparrow@pirates.sea',
            'kate.bishop@avenger.org', 'liam.neeson@taken.com', 'mary.poppins@nanny.uk',
            'nancy.drew@detective.com', 'oscar.wilde@writer.ie', 'petra.parker@spider.web',
            'quinn.fabray@glee.tv', 'rachel.green@friends.tv'
        ],
        'phone': [
            '555-1234', '555-2345', '555-3456', '555-4567', '555-5678',
            '555-6789', '555-7890', '555-8901', '555-9012', '555-0123',
            '555-1357', '555-2468', '555-3579', '555-4680', '555-5791',
            '555-6802', '555-7913', '555-8024', '555-9135', '555-0246'
        ],
        'ssn': [
            '123-45-6789', '234-56-7890', '345-67-8901', '456-78-9012', '567-89-0123',
            '678-90-1234', '789-01-2345', '890-12-3456', '901-23-4567', '012-34-5678',
            '123-56-7890', '234-67-8901', '345-78-9012', '456-89-0123', '567-90-1234',
            '678-01-2345', '789-12-3456', '890-23-4567', '901-34-5678', '012-45-6789'
        ],
        'credit_card': [
            '4532-1234-5678-9012', '5425-2345-6789-0123', '3782-3456-7890-1234',
            '6011-4567-8901-2345', '4916-5678-9012-3456', '5234-6789-0123-4567',
            '3714-7890-1234-5678', '6221-8901-2345-6789', '4539-9012-3456-7890',
            '5310-0123-4567-8901', '3787-1234-5678-9012', '6011-2345-6789-0123',
            '4485-3456-7890-1234', '5412-4567-8901-2345', '3796-5678-9012-3456',
            '6331-6789-0123-4567', '4929-7890-1234-5678', '5523-8901-2345-6789',
            '3759-9012-3456-7890', '6062-0123-4567-8901'
        ]
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = pd.read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if df[col].dtype == 'object':
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
        print(f"    Sample: {df[col].iloc[0]}")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure field name** for masking

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Customize this!
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_name": "email"  # Field to mask - CUSTOMIZE THIS!
    }

kwargs = create_config_kwargs()
field_name = kwargs["field_name"]

# Validate field exists
print(f"\n🔍 Validating field configuration...\n")
if field_name not in df.columns:
    raise ValueError(
        f"❌ Field '{field_name}' not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'field_name' in create_config_kwargs()"
    )

print(f"✓ Field to mask: '{field_name}'")
print(f"  Unique values: {df[field_name].nunique()}")
print(f"  Sample values:")
for i, val in enumerate(df[field_name].head(5), 1):
    print(f"    {i}. {val}")
print(f"  Average length: {df[field_name].str.len().mean():.1f} characters")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="masking",
    description=f"Partial masking of '{field_name}' field using position-based strategy.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Processing {field_name} field",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create PartialMaskingOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `field_name='{field_name}'`: Field to mask
- `mode='REPLACE'`: Modify original field (or 'CREATE' for new field)
- `mask_strategy='fixed'`: Position-based masking strategy
- `unmasked_prefix=3`: Keep first 3 characters visible
- `unmasked_suffix=10`: Keep last 10 characters visible (e.g., @domain.com)
- `mask_char='*'`: Character used for masking
- `preserve_separators=True`: Keep '@', '.', '-' characters
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = PartialMaskingOperation(
    # Core parameters
    field_name=field_name,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Masking strategy parameters
    mask_strategy='fixed',           # Position-based masking
    mask_char='*',                   # Masking character
    unmasked_prefix=3,               # Keep first 3 chars
    unmasked_suffix=10,              # Keep last 10 chars (domain)
    
    # Format preservation
    preserve_separators=True,        # Keep @, ., - visible
    preserve_word_boundaries=False,
    
    # Text processing
    case_sensitive=True,
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Field:            {operation.field_name}")
print(f"  Strategy:         {operation.mask_strategy}")
print(f"  Mask char:        '{operation.mask_char}'")
print(f"  Unmasked prefix:  {operation.unmasked_prefix}")
print(f"  Unmasked suffix:  {operation.unmasked_suffix}")
print(f"  Preserve seps:    {operation.preserve_separators}")
print(f"  Visualizations:   {operation.generate_visualization}")
print(f"  Save output:      {operation.save_output}")
print(f"  Force recalc:     {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing partial masking...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the masked output from task_dir
- Compare with original data
- Review metrics and artifacts

**Generated artifacts:**
- Masked CSV in output/
- Metrics JSON in metrics/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Masked Records:")
    print(f"\n{'Original':<40} → {'Masked':<40}")
    print("-" * 82)
    for orig, masked in zip(df[field_name].head(10), result_df[field_name].head(10)):
        print(f"{orig:<40} → {masked:<40}")
    
    # Calculate masking statistics
    total_chars_original = df[field_name].str.len().sum()
    total_chars_masked = result_df[field_name].apply(
        lambda x: str(x).count('*')
    ).sum()
    mask_percentage = (total_chars_masked / total_chars_original) * 100 if total_chars_original > 0 else 0
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Masked records:     {len(result_df)}")
    print(f"  Characters masked:  {mask_percentage:.1f}%")
    print(f"  Prefix preserved:   {operation.unmasked_prefix} chars")
    print(f"  Suffix preserved:   {operation.unmasked_suffix} chars")
    
    # Display result metrics
    if result.metrics:
        print("\n🔒 Privacy Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Partial masking preserves data utility while protecting sensitive information!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Masked CSV
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST)
metrics_files = list((task_dir / 'metrics').glob('*.json'))
if metrics_files:
    # 1) Exclude data_types inside IF
    filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

    if filtered:
        # Use only filtered files
        target_files = filtered
        print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
    else:
        # Fallback: nothing left after filtering → use all
        target_files = metrics_files
        print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

    # 2) Pick latest
    target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_metrics_file = target_files[0]

    print(f"📄 Loading LATEST: {latest_metrics_file.name}")
    mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
    print("=" * 80)
    
    with open(latest_metrics_file) as f:
        data = json.load(f)
    
    # Display key metrics
    if 'metrics' in data:
        metrics = data['metrics']
        print(f"\n📈 Key Masking Metrics:")
        
        # Masking effectiveness
        if 'partial_mask_rate' in metrics:
            print(f"   Mask Rate:                 {metrics['partial_mask_rate']:.2%}")
        if 'average_visibility' in metrics:
            print(f"   Avg Visibility:            {metrics['average_visibility']:.2%}")
        if 'values_masked' in metrics:
            print(f"   Values Masked:             {metrics['values_masked']}")
        if 'suppression_rate' in metrics:
            print(f"   Suppression Rate:          {metrics['suppression_rate']:.2%}")
        
        # Strategy info
        print(f"\n🎯 Strategy Details:")
        print(f"   Operation Type:            {metrics.get('operation_type', 'N/A')}")
        print(f"   Mask Strategy:             {metrics.get('mask_strategy', 'N/A')}")
        print(f"   Pattern Type:              {metrics.get('pattern_type') or 'N/A'}")
        print(f"   Preserve Separators:       {metrics.get('preserve_separators')}")
        print(f"   Consistency Fields Count:  {metrics.get('consistency_fields_count', 0)}")
        print(f"   Consistency Fields Proc:   {metrics.get('consistency_fields_processed', 'N/A')}")
        
        # Strategy distribution
        if 'masking_strategy_distribution' in metrics:
            dist = metrics['masking_strategy_distribution']
            print(f"\n🔬 Strategy Distribution:")
            for k, v in dist.items():
                print(f"   {k.capitalize():<15}: {v}")
        
        # Field info section
        if 'field_info' in metrics:
            fi = metrics['field_info']
            print(f"\n📊 Field Info:")
            orig = fi.get('original', {})
            proc = fi.get('processed', {})
            
            print("   Original:")
            print(f"      Total Count:            {orig.get('total_count', 'N/A')}")
            print(f"      Null Count:             {orig.get('null_count', 'N/A')}")
            print(f"      Unique Count:           {orig.get('unique_count', 'N/A')}")
            print(f"      Uniqueness Ratio:       {orig.get('uniqueness_ratio', 0):.2%}")
            
            print("   Processed:")
            print(f"      Total Count:            {proc.get('total_count', 'N/A')}")
            print(f"      Null Count:             {proc.get('null_count', 'N/A')}")
            print(f"      Unique Count:           {proc.get('unique_count', 'N/A')}")
            print(f"      Uniqueness Ratio:       {proc.get('uniqueness_ratio', 0):.2%}")
        
        # Masking summary block
        if 'masking' in metrics:
            m = metrics['masking']
            print(f"\n🛡️ Masking Summary:")
            print(f"   Total Records:             {m.get('total_records', 'N/A')}")
            print(f"   Masked Records:            {m.get('masked_records', 'N/A')}")
            print(f"   Masking Ratio:             {m.get('masking_ratio', 0):.2%}")
            print(f"   Null Count:                {m.get('null_count', 'N/A')}")
        
        # Timestamp
        if 'timestamp' in metrics:
            print(f"\n⏱ Timestamp:                  {metrics['timestamp']}")
else:
    print("⚠️  No metrics files found.")

# 2. OUTPUT DATA (NEWEST)
output_files = list((task_dir / 'output').glob('*.csv'))
if output_files:
    output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest = output_files[0]
    print(f"\n\n2️⃣ OUTPUT DATA (LATEST): {latest.name}")
    print("-" * 80)
    
    output_df = pd.read_csv(latest)
    print(f"\n📋 First 10 Masked Records:")
    print(output_df.head(10))
    
    # Show masking pattern analysis
    if field_name in output_df.columns:
        print(f"\n📊 Masking Pattern Analysis:")
        print("-" * 60)

        # FIXED: use raw string for regex
        masked_lengths = output_df[field_name].str.count(r'\*')

        print(f"   Avg masked chars:   {masked_lengths.mean():.1f}")
        print(f"   Min masked chars:   {masked_lengths.min()}")
        print(f"   Max masked chars:   {masked_lengths.max()}")
else:
    print("⚠️  No output data files found.")

# 3. VISUALIZATIONS (NEWEST BATCH)
viz_files = list((task_dir / 'visualizations').glob('*.png'))
if viz_files:
    viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_time = viz_files[0].stat().st_mtime
    latest_batch = [f for f in viz_files if abs(f.stat().st_mtime - latest_time) < 10]
    
    print(f"\n\n3️⃣ VISUALIZATIONS (LATEST BATCH):")
    print("-" * 80)
    print(f"Found {len(latest_batch)} visualization(s) from latest operation\n")
    
    for viz_file in latest_batch:
        print(f"\n{viz_file.stem.replace('_', ' ').title()}")
        display(Image(filename=str(viz_file)))
else:
    print("⚠️  No visualization files found.")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ **Load data**: Read CSV from examples/data_examples/  
✅ **Setup environment**: TaskReporter, DataSource, ProgressTracker  
✅ **Configure operation**: Full production-style parameters with field config  
✅ **Execute**: Use operation.execute() with explicit configs  
✅ **Analyze results**: Review metrics and artifacts

**Key takeaways:**
- `operation.execute()` handles end-to-end processing
- Position-based masking preserves prefix and suffix while hiding middle
- `preserve_separators=True` keeps special characters (@, ., -) visible
- Field name is configurable via `create_config_kwargs()`
- Partial masking balances privacy protection with data utility
- All artifacts saved in structured directories

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_partial_masking_advanced.ipynb`** includes:
- All masking strategies (fixed, pattern, random, words)
- Pattern-based masking (email, phone, SSN, credit card)
- Random percentage masking
- Word boundary preservation
- Multi-field consistency masking
- Preset-based masking templates
- Custom regex patterns

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Masking Strategies Guide](../../docs/masking_strategies.md)